In [13]:
import sys
sys.path.append(r'C:\Users\aweso\OneDrive\Documents\LLM_zoomcamp\intro')

In [14]:
print(123)

123


In [15]:
import os
from dotenv import load_dotenv
from groq import Groq

# Load environment variables
load_dotenv()

# Configure Groq
api_key = os.getenv("GROQ_API_KEY")
client = Groq(api_key=api_key)


In [16]:
messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

response = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=messages,
)

print(response.choices[0].message.content)

I'm happy to help, but I'm a large language model, I don't have real-time access to specific courses or programs. Could you please provide more context or information about the course you're trying to join? This will help me better understand your question and provide a more accurate response.


In [17]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [18]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [19]:
search_tool = {
        "type": "function",
        "function": {
            "name": "search_faq",          
            "description": "Search the FAQ documents for an answer",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "The search query"
                    }
                },
                "required": ["query"]
            }
        }
    }

In [20]:
response = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=messages,
    tools=[search_tool]
)

if response.choices[0].message.tool_calls:
    print(response.choices[0].message.tool_calls[0].function)
else:
    print("No tool calls. Content:", response.choices[0].message.content)

Function(arguments='{"query":"joining the course"}', name='search_faq')


In [21]:
call = response.choices[0].message.tool_calls[0].function

In [22]:
import json

args = json.loads(call.arguments)

In [23]:
call.name

'search_faq'

In [24]:
results = search(**args)

In [25]:
result_json = json.dumps(results, indent=2)

In [26]:
messages.append(response.choices[0].message)

tool_call = response.choices[0].message.tool_calls[0]
messages.append({
    "role": "tool",
    "tool_call_id": tool_call.id,
    "name": tool_call.function.name,
    "content": result_json
})

In [27]:
messages

[{'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ChatCompletionMessage(content=None, role='assistant', annotations=None, executed_tools=None, function_call=None, reasoning=None, tool_calls=[ChatCompletionMessageToolCall(id='wt7tq14v7', function=Function(arguments='{"query":"joining the course"}', name='search_faq'), type='function')]),
 {'role': 'tool',
  'tool_call_id': 'wt7tq14v7',
  'name': 'search_faq',
  'content': '[\n  {\n    "id": "04919992b3",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "How should I start the course and follow the weekly workflow?",\n    "answer": "Start with the [LLM Zoomcamp docs](https://datatalks.club/docs/courses/llm-zoomcamp/), the [general Zoomcamp logistics docs](https://datatalks.club/docs/courses/zoomcamp-logistics/), and the [LLM Zoomcamp GitHub repository](https://github.com/DataTalksClub/llm-zoomcamp).\\n\\nYou can start whenever you want. The videos and GitHub 

In [28]:
response = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=messages,
    tools=[search_tool]
)
response.choices[0].message.content

In [29]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

In [30]:
def make_call(call):
        args = json.loads(call.function.arguments)
    
        if call.function.name == "search_faq":
            result = search(**args)
        else:
            result = []
    
        result_json = json.dumps(result, indent=2)
    
        return {
            "role": "tool",
            "tool_call_id": call.id,
            "name": call.function.name,
            "content": result_json,
        }

In [31]:
question = "I just discovered the course. Can I join it?"

messages = [
    {"role": "system", "content": instructions},
    {"role": "user", "content": question},
]

response = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=messages,
    tools=[search_tool]
)


messages.append(response.choices[0].message)
has_function_calls = False

if response.choices[0].message.tool_calls:
    for tool_call in response.choices[0].message.tool_calls:
        print("function_call:", tool_call.function.name, tool_call.function.arguments)
        call_output = make_call(tool_call)
        messages.append(call_output)
        has_function_calls = True
else:
        print("ASSISTANT:")
        print(response.choices[0].message.content)

function_call: search_faq {"query":"join course"}
function_call: search_faq {"query":"how to join course online"}
function_call: search_faq {"query":"course enrollment process"}
function_call: search_faq {"query":"course availability deadline"}


In [32]:
it = 1

while True:
    print(f"iteration #{it}...")
    has_function_calls = False

    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=messages,
        tools=[search_tool],
    )

    message = response.choices[0].message
    messages.append(message)

    if message.tool_calls:
        for tool_call in message.tool_calls:
            print("function_call:", tool_call.function.name, tool_call.function.arguments)
            call_output = make_call(tool_call)
            messages.append(call_output)
            has_function_calls = True

    elif message.content:
        print("ASSISTANT:")
        print(message.content)

    it = it + 1
    if has_function_calls == False:
        break


iteration #1...
ASSISTANT:
You are able to join the course, but if you want to receive a certificate, you need to submit your project while it's still being accepted. However, you can still work through the material and prepare your project in self-paced mode. 

To get the certificate, you need to finish a capstone project and complete the required peer reviews. Homework is not required. Once the homework submission form is closed, you can no longer submit it. 

The videos and GitHub materials are available, and the deadlines are listed in the course management platform. A typical workflow is to watch the lesson videos, work through the lesson notebooks/code, read the homework instructions on GitHub, and submit answers through the course platform before the deadline. 

The course will be offered again in Summer 2027. 

Is there anything else you would like to explore?


In [33]:
def agent_loop(instructions, question, model="llama-3.1-8b-instant") -> str:
    messages = [
        {"role": "system", "content": instructions},
        {"role": "user", "content": question}
    ]

    it = 1
    last_answer = ""

    while True:
        print(f"iteration #{it}...")
        has_function_calls = False

        response = client.chat.completions.create(
            model=model,
            messages=messages,
            tools=[search_tool]
        )

        message = response.choices[0].message
        messages.append(message)

        if message.tool_calls:
            for tool_call in message.tool_calls:
                print("function_call:", tool_call.function.name, tool_call.function.arguments)
                call_output = make_call(tool_call)
                messages.append(call_output)
                has_function_calls = True

        elif message.content:
            print("ASSISTANT:")
            last_answer = message.content
            print(message.content)

        it = it + 1
        if not has_function_calls:
            break

    return last_answer

In [34]:
agent_loop(instructions, "How do I run Olama locally?")

iteration #1...
function_call: search_faq {"query":"how to run Olama locally on my computer"}
function_call: search_faq {"query":"steps to deploy Olama locally"}
function_call: search_faq {"query":"Olama installation guide for local environment"}
iteration #2...
ASSISTANT:
To run Olama locally, you can follow these steps:

1. Install Ollama by visiting [https://ollama.com/download](https://ollama.com/download) and choosing your operating system.
2. Once installed, open a terminal and type `ollama run llama3` to start the model locally.
3. You can test the Ollama local server by running the command `curl http://localhost:11434`.
4. To use Ollama with Python, install the Python client with `pip install ollama` and use the following example code to send a chat request:
   ```python
import ollama

response = ollama.chat(
    model='llama3',
    messages=[
        {'role': 'user', 'content': 'your_prompt'}
    ]
)

print(response['message']['content'])
```

You can also use other LLM provid

"To run Olama locally, you can follow these steps:\n\n1. Install Ollama by visiting [https://ollama.com/download](https://ollama.com/download) and choosing your operating system.\n2. Once installed, open a terminal and type `ollama run llama3` to start the model locally.\n3. You can test the Ollama local server by running the command `curl http://localhost:11434`.\n4. To use Ollama with Python, install the Python client with `pip install ollama` and use the following example code to send a chat request:\n   ```python\nimport ollama\n\nresponse = ollama.chat(\n    model='llama3',\n    messages=[\n        {'role': 'user', 'content': 'your_prompt'}\n    ]\n)\n\nprint(response['message']['content'])\n```\n\nYou can also use other LLM providers like OpenAI, Gemini, Anthropic, xAI, Grok, or local models served through Ollama or LM Studio with Kestra's AI plugin.\n\nAs for running the course locally, you can use Python, Jupyter, and Docker to set up the environment. You can also use alternati

In [35]:
agent_loop(instructions, "I just discovered the course. Can I still join it?")

iteration #1...
function_call: search_faq {"query":"joining course late"}
function_call: search_faq {"query":"late course enrollment"}
iteration #2...
ASSISTANT:
Based on the search results, it appears that you can still join the course, but if you want to receive a certificate, you need to submit your project while the course is still accepting submissions.


'Based on the search results, it appears that you can still join the course, but if you want to receive a certificate, you need to submit your project while the course is still accepting submissions.'

In [36]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "what's queen gambit?")

iteration #1...
function_call: search_faq {"query":"Queen Gambit course"}
function_call: search_faq {"query":"Queen Gambit"}
function_call: search_faq {"query":"gambit course"}
function_call: search_faq {"query":"gambit"}
iteration #2...
ASSISTANT:
It seems the question about "Queen Gambit" is not related to the course. I'm happy to help you with any other questions you may have about the course or its logistics. 

Are there other areas that you would like to explore?


'It seems the question about "Queen Gambit" is not related to the course. I\'m happy to help you with any other questions you may have about the course or its logistics. \n\nAre there other areas that you would like to explore?'

In [37]:
!uv add toyaikit

Resolved 186 packages in 3.22s
Prepared 7 packages in 3.35s
Installed 7 packages in 904ms
 + anthropic==0.117.0
 + docstring-parser==0.18.0
 + genai-prices==0.0.71
 + httpcore2==2.3.0
 + httpx2==2.3.0
 + toyaikit==0.0.11
 + truststore==0.10.4


In [73]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

In [59]:
import os
from openai import OpenAI
from toyaikit.llm import OpenAIChatCompletionsClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback, OpenAIChatCompletionsRunner

In [62]:
 # Point the OpenAI client to Groq's API
groq_openai_client = OpenAI(
    api_key=os.environ.get("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
    )
    

In [63]:
llm_client = OpenAIClient(client=groq_openai_client)

In [64]:
agent_tools = Tools()
agent_tools.add_tool(search, search_tool)

In [65]:
def search(query: str) -> dict[str, str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={"question": 3.0, "section": 0.5},
        filter_dict={"course": "llm-zoomcamp"}
    )

In [66]:
agent_tools = Tools()
agent_tools.add_tool(search)

In [67]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the FAQ database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [74]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)
    
runner = OpenAIChatCompletionsRunner(
        tools=agent_tools,
        developer_prompt=instructions,
        chat_interface=chat_interface,
        llm_client=OpenAIChatCompletionsClient(
            client=client,
            model="llama-3.3-70b-versatile"   # or llama-3.1-8b-instant
        )
    )

In [77]:
result = runner.loop(
    prompt="How do I run Olama locally?",
    callback=callback,
)

-> Response received


-> Response received


-> Response received


c:\Users\aweso\OneDrive\Documents\LLM_zoomcamp\.venv\Lib\site-packages\toyaikit\chat\runners.py:542: UnknownModelWarning: No pricing data for model 'llama-3.3-70b-versatile'. Register it with PricingConfig.register_model(...) to get cost calculations.
  cost = self.pricing_config.calculate_cost(


In [78]:
result.cost

In [79]:
result.all_messages

[{'role': 'system',
  'content': "You're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches.\n\nTry to expand your search by using new keywords\nbased on the results you get from the search.\n\nAt the end, ask if there are other areas that the user wants to explore."},
 {'role': 'user', 'content': 'How do I run Olama locally?'},
 ChatCompletionMessage(content=None, role='assistant', annotations=None, executed_tools=None, function_call=None, reasoning=None, tool_calls=[ChatCompletionMessageToolCall(id='zpgpf1dy7', function=Function(arguments='{"query":"Olama local installation guide"}', name='search'), type='function'), ChatCompletionMessageToolCall(id='cm5vh0wpm', function=Function(arguments='{"query":"Olama installation requirements"}', name='search'), type

In [80]:
result2 = runner.loop(
    prompt="How do I run a different model?",
    previous_messages=result.all_messages,
    callback=callback,
)

-> Response received


-> Response received


c:\Users\aweso\OneDrive\Documents\LLM_zoomcamp\.venv\Lib\site-packages\toyaikit\chat\runners.py:542: UnknownModelWarning: No pricing data for model 'llama-3.3-70b-versatile'. Register it with PricingConfig.register_model(...) to get cost calculations.
  cost = self.pricing_config.calculate_cost(


In [ ]:
runner.run()